In [1]:
from typing import TypedDict, Annotated
from langchain_core.messages import (HumanMessage, AIMessage, ToolMessage )
from langchain_core.tools import tool
from langgraph.graph import ( StateGraph, START, END)
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq

In [2]:
import os
from dotenv import load_dotenv

# Load API key from backend/.env
dotenv_path = os.path.abspath("../../../../.env")
load_dotenv(dotenv_path)

if not os.environ.get("GROQ_API_KEY"):
    print("Warning: GROQ_API_KEY not found in backend/.env")
else:
    print("GROQ_API_KEY loaded successfully.")

GROQ_API_KEY loaded successfully.


In [3]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [ ]:
class SourceFile(TypedDict):
    """Metadata for a single raw source file."""
    path: str
    filename: str
    extension: str
    size_bytes: int
    category: str  # catalog | review | document | image | promotion | specialty


In [ ]:
class WikiState(TypedDict):
    """
    Central state that flows through all LangGraph nodes.

    Both 'knowledge' and 'marketing' sections use this same state.
    The `wiki_section` field controls which sub-directory is targeted.
    """

    # ── Identity ──
    merchant_id: str
    wiki_section: str  # "knowledge" | "marketing"

    # ── Paths ──
    source_dir: str  # where uploaded files are
    wiki_base_path: str  # root of merchant_knowledge/

    # ── Source scanning ──
    uploaded_files: list[SourceFile]
    classified_sources: dict[str, list[SourceFile]]  # category -> files

    # # ── Extraction ──
    # extracted_products: list[ExtractedProduct]
    # extracted_reviews: list[ExtractedReview]
    # review_syntheses: list[ReviewSynthesis]

    # # ── Wiki maintenance ──
    # entities: list[dict[str, Any]]
    # existing_pages: dict[str, str]  # slug -> file_path
    # pages_to_create: list[WikiPage]
    # pages_to_update: list[WikiPage]
    # contradictions: list[Contradiction]

    # # ── Output ──
    # generated_pages: list[WikiPage]
    # index_updates: list[dict[str, str]]
    # log_entry: LogEntry
    # validation_result: ValidationResult

    # # ── Control ──
    # validation_errors: list[str]
    # status: str  # running | success | failed
    # error: str


In [ ]:
"""
scan_sources — DETERMINISTIC NODE

Walks the source directory, identifies all files with metadata.
Copies them into raw/ (immutable) if not already there.
"""

import os
from pathlib import Path
from datetime import datetime

# from ..state import WikiState, SourceFile
# from ..utils.file_io import copy_to_raw, list_files


def scan_sources(state: WikiState) -> dict:
    """
    Scan the source directory for all uploadable files.
    Copy each file into raw/{section}/ as immutable source.
    """
    source_dir = state["source_dir"]
    wiki_base = state["wiki_base_path"]
    section = state.get("wiki_section", "knowledge")

    if not os.path.exists(source_dir):
        return {
            "uploaded_files": [],
            "status": "failed",
            "error": f"Source directory not found: {source_dir}",
        }

    # Supported extensions
    supported = [".csv", ".xlsx", ".xls", ".pdf", ".txt", ".json", ".md"]

    # Find all files
    raw_files = list_files(source_dir, extensions=supported)

    uploaded: list[SourceFile] = []

    for file_info in raw_files:
        filepath = file_info["path"]
        ext = file_info["extension"]

        # Determine raw category based on extension
        if ext in (".csv", ".xlsx", ".xls"):
            category = "catalog"  # Will be reclassified by classify_sources
        elif ext == ".pdf":
            category = "documents"
        elif ext in (".txt", ".json", ".md"):
            category = "documents"
        else:
            category = "documents"

        # Copy to raw/ (immutable)
        raw_dest = copy_to_raw(
            filepath,
            os.path.join(wiki_base, "raw", section),
            category,
        )

        uploaded.append(SourceFile(
            path=raw_dest,
            filename=file_info["filename"],
            extension=ext,
            size_bytes=file_info["size_bytes"],
            category=category,
        ))

    print(f" Scanned {len(uploaded)} files from {source_dir}")
    return {
        "uploaded_files": uploaded,
        "status": "running",
    }


In [ ]:
graph = StateGraph(WikiState)

# nodes
graph.add_node("collect_data", collect_data)


# edges
# graph.add_edge(START, "merchant_llm" )
# graph.add_conditional_edges("merchant_llm", tools_condition)
# graph.add_edge("tools", "merchant_llm")


workflow = graph.compile()

In [ ]:
#  tool binding to the llm
# llm_with_tools = llm.bind_tools(tools)

In [ ]:
# Prompt
SYSTEM_PROMPT = """
You are the Merchant Commerce Agent.

You are responsible for handling customer shopping requests.

You have access to commerce tools.

Rules:

1. Never claim that a product was added unless add_to_cart
   successfully confirms it.

2. Never directly modify the cart yourself.

3. Use get_cart when you need current cart information.

4. Use get_upsell_products when the customer asks for
   recommendations or when an appropriate complementary
   product can be suggested.

5. Do not invent products.

6. Keep responses concise and commerce-focused.

7. If a tool fails, clearly tell the customer that the
   requested operation could not be completed.

8. MUST: Do not add the item to the cart until and unless customer ask to do, if he is asking about the product just tell it  where you have or not,
    and then ask him if he would like it to be added to the cart and along with it recommend the items that this inventory have
"""

In [ ]:
# class CommerceState(TypedDict):

#     messages: Annotated[ list, add_messages ]

# """ later add ons
# user_id
# merchant_id
# cart
# payment_context
# policy_context
# """

In [ ]:
# def merchant_llm_node(state: CommerceState):

#     messages = state["messages"]

#     system_message = {
#         "role": "system",
#         "content": SYSTEM_PROMPT
#     }

#     response = llm_with_tools.invoke(
#         [system_message] + messages
#     )

#     return {
#         "messages": [response]
#     }

    

In [ ]:
# tool_node = ToolNode(tools)

In [ ]:
# graph = StateGraph(CommerceState)

# # nodes
# graph.add_node("merchant_llm", merchant_llm_node )
# graph.add_node( "tools", tool_node )

# # edges
# graph.add_edge(START, "merchant_llm" )
# graph.add_conditional_edges("merchant_llm", tools_condition)
# graph.add_edge("tools", "merchant_llm")


# workflow = graph.compile()